# Mission 03: ACID and ETL Control Room

A finance team loads daily transaction files from an operational system into a reporting database. The process must **never lose money and never count a transaction twice**, even when a file is loaded again or contains a bad record.

**Lessons used:** 03 (constraints), 09 (Python + SQL), 11 (transactions), 16 (roles), 17 (triggers), 18 (backup and restore).

The transactions are **synthetic** (generated below).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import create_engine, text

sys.path.insert(0, str(Path.cwd().parent.parent / "tools"))
import datasets                                   # course helper: downloads and loads the sample data

engine = create_engine(datasets.ensure_chinook())  # starts PostgreSQL in Docker and loads Chinook on first use

def q(sql: str, **params) -> pd.DataFrame:
    """Run a SQL query and return the result as a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

lab = create_engine(datasets.POSTGRES_URL.format(db="course"))   # a separate database for experiments

def run(sql: str) -> None:
    """Execute SQL statements (DDL/DML) in the lab database, in one transaction."""
    with lab.begin() as conn:
        if "$$" in sql:                       # function bodies contain ";" -> send the whole script at once
            conn.execute(text(sql))
            return
        for stmt in [s for s in sql.split(";") if s.strip()]:
            conn.execute(text(stmt))

def lq(sql: str, **params) -> pd.DataFrame:
    with lab.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

import numpy as np

def make_batch(day: str, n: int, seed: int, bad_withdrawal: bool = False) -> pd.DataFrame:
    """A synthetic daily transaction file: deposits (+) and withdrawals (-) for 20 accounts."""
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({"txn_id": [f"{day}-{i:05d}" for i in range(n)], "account_id": rng.integers(1, 21, n),
                       "amount": rng.normal(5, 40, n).round(2), "txn_date": pd.Timestamp(day)})
    if bad_withdrawal:
        df.loc[n - 1, ["account_id", "amount"]] = [7, -1_000_000]     # would overdraw account 7
    return df

batches = {"2026-01-05": make_batch("2026-01-05", 300, 1), "2026-01-06": make_batch("2026-01-06", 280, 2),
           "2026-01-07": make_batch("2026-01-07", 310, 3, bad_withdrawal=True)}
batches["2026-01-05"].head()

## Tasks

1. **Schema.** In a schema `finance`, create `account` (balance may never be negative), `ledger` (one row per transaction; the `txn_id` must be unique) and `load_batch` (one row per loaded file, with its status). Every account starts with a balance of 1,000.
2. **Idempotent load.** Write `load(day, df)`: load one batch into `ledger` and update the balances **in one transaction**. Loading the same file twice must not change anything.
3. **Failure handling.** Load the three batches in order, then load `2026-01-06` again. The bad batch must leave **no** trace in `ledger` and `account`, and `load_batch` must record it as `failed` with the error.
4. **Append-only ledger.** Add a trigger that rejects `UPDATE` and `DELETE` on `ledger`. Corrections must be new rows.
5. **Reconciliation.** Prove that for every account: opening balance + sum of ledger amounts = current balance.
6. **Roles.** Create a role `etl_loader` that can load data but not delete or read anything else, and a role `finance_reader` that can only read. Show one allowed and one denied action for each.
7. **Backup.** Back up the `finance` schema with `pg_dump`, restore it into a new database, and verify the totals. Write 3–5 sentences on RPO/RTO for this pipeline.

In [ ]:
# Task 1
...

In [ ]:
# Task 2
def load(day: str, df: pd.DataFrame) -> str:
    """Load one batch idempotently in one transaction; return 'loaded', 'skipped' or 'failed: ...'."""
    ...

In [ ]:
# Tasks 3-7
...

## Submission Checklist

- [ ] A repeated load changes nothing; a failed load leaves nothing behind.
- [ ] Constraints and a trigger protect the business rules.
- [ ] Reconciliation passes for every account.
- [ ] Roles follow least privilege, demonstrated with allowed and denied actions.
- [ ] A restore was tested, and RPO/RTO are stated.

A solution is in `03-ACID-and-ETL-Control-Room-solution.ipynb`. Try the mission first.